<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from mnist import MNIST
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# If randomness changes between runs, width is no longer the only thing changing.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer GPU when available, but keep the workflow fully CPU-compatible.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# A batch of 512 is large enough for stable updates and still cheap on MNIST.
BATCH_SIZE = 512

# This is the experiment: change hidden width, keep the rest of the recipe fixed.
HIDDEN_SIZES = [128, 256, 512]

# Ten epochs are sufficient to compare convergence under this fixed protocol.
EPOCHS = 10

# One learning rate for all widths keeps optimization from becoming a hidden second experiment.
LEARNING_RATE = 1e-3

# Add a small weight penalty so larger models cannot grow weights freely just because they have more capacity.
WEIGHT_DECAY = 1e-3

# MNIST defines ten digit classes and 28x28 grayscale inputs.
N_CLASSES = 10
# MNIST images are 28×28 pixels; keep this explicit because model input size depends on it.
IMAGE_SIZE = 28
# Flatten each image to 784 features because this lab uses an MLP rather than a CNN.
INPUT_DIM = IMAGE_SIZE * IMAGE_SIZE

# Put every figure in one predictable folder so the experiment can be rerun and checked automatically.
OUTPUT_DIR = Path("../outputs/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

PyTorch version: 2.14.0+cu130
Device: cpu


## 1. Validate MNIST Data and Output Paths


In [2]:
# Verify the exact local MNIST files before constructing any dataset object.
DATA_DIR = Path("../data")

# Declare the exact IDX files needed so environment validation can fail before training starts.
REQUIRED_MNIST_FILES = [
    "train-images-idx3-ubyte",
    "train-labels-idx1-ubyte",
    "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte",
]

# Collect all missing IDX files at once so setup failures are reported together.
missing_files = [
    name
    for name in REQUIRED_MNIST_FILES
    if not (DATA_DIR / name).exists()
]

# Stop before dataset construction so missing IDX files cannot fail later ambiguously.
if missing_files:
    raise FileNotFoundError(
        "Missing MNIST files: " + ", ".join(missing_files)
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MNIST data directory: {DATA_DIR}")
print(f"Figures directory: {OUTPUT_DIR}")
print("All required MNIST files are present.")

MNIST data directory: ../data
Figures directory: ../outputs/figures
All required MNIST files are present.


## 2. Build the MNIST Dataset and Mini-Batch Pipeline


In [3]:
class MNISTDataset(Dataset):
    """Turn the raw MNIST files into samples PyTorch can train on.

    The original IDX files store each digit as 784 byte values. We expose one
    sample at a time, normalize its pixels to [0, 1], and keep the label as an
    integer class index.

    Doing the normalization here gives every model exactly the same input
    representation before architecture differences enter the experiment.
    """

    image_size = IMAGE_SIZE

    def __init__(self, partition, mnist_dir):
        """Load either the training split or the test split — nothing ambiguous.

        python-mnist exposes those two parser names directly, so accepting only
        "training" and "testing" keeps the dataset contract explicit and catches
        typos before they turn into harder-to-read file errors.
        """
        # Only the two parser names provided by python-mnist are accepted.
        if partition not in ("training", "testing"):
            raise ValueError(
                f"{partition!r} is not a valid partition."
            )

        mnist = MNIST(str(mnist_dir))
        parser = getattr(mnist, f"load_{partition}")

        self.images, self.labels = parser()
        self.nimages = len(self.images)

    def __len__(self):
        """Tell PyTorch how many samples are available in this split."""
        return self.nimages

    def __getitem__(self, index):
        """Prepare one digit for the network.

        MNIST pixels arrive as integers from 0 to 255. Dividing by 255 moves
        them to [0, 1], which gives optimization a stable, consistent scale.

        The image stays flattened because this experiment uses an MLP, not a
        convolutional network.
        """
        # Normalize at the dataset boundary so every model receives exactly the same numeric input scale.
        image = (
            np.asarray(
                self.images[index],
                dtype=np.float32,
            )
            / 255.0
        )

        label = int(self.labels[index])

        return {
            "image": image,
            "label": label,
        }

    @staticmethod
    def collate_fn(data_batch):
        """Turn a list of individual samples into one training batch.

        Images are stacked into a single float32 matrix. Labels become int64
        because CrossEntropyLoss expects class indices, not one-hot vectors.

        Keeping this conversion in one place prevents subtle dtype differences
        between training and evaluation.
        """
        # Stack individual normalized samples into one contiguous mini-batch array.
images = np.stack(
            [item["image"] for item in data_batch],
            axis=0,
        ).astype(np.float32)

        # Keep labels as int64 class indices because CrossEntropyLoss consumes integer targets.
labels = np.asarray(
            [item["label"] for item in data_batch],
            dtype=np.int64,
        )

        return {
            "image": images,
            "label": labels,
        }


def build_dataset_and_loader(
    batch_size,
    partition,
    data_dir,
):
    """Build the dataset and the iterator that will feed the model.

    Training samples are shuffled because seeing examples in a new order helps
    mini-batch optimization. Test samples stay in a fixed order because there
    is nothing to learn during evaluation.

    num_workers=0 is intentional here: the dataset is small, and a single
    process makes the notebook behave the same way across operating systems.
    """
    dataset = MNISTDataset(
        partition=partition,
        mnist_dir=data_dir,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        # Shuffle only training data; deterministic test order simplifies diagnostics.
        shuffle=(partition == "training"),
        # Single-process loading maximizes notebook portability and reproducibility.
        num_workers=0,
        collate_fn=dataset.collate_fn,
    )

    return dataset, loader

print("MNIST dataset and DataLoader pipeline defined.")

MNIST dataset and DataLoader pipeline defined.


## 3. Load and Validate Training and Testing Data


In [4]:
# Build both splits through the same pipeline; only shuffling should differ between train and test.
train_dataset, train_loader = build_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="training",
    data_dir=DATA_DIR,
)

test_dataset, test_loader = build_dataset_and_loader(
    batch_size=BATCH_SIZE,
    partition="testing",
    data_dir=DATA_DIR,
)

# Before trusting any model result, make sure the benchmark itself still matches canonical MNIST.
# Verify the canonical MNIST training cardinality before comparing models.
if len(train_dataset) != 60_000:
    raise ValueError(
        f"Expected 60,000 training images; found {len(train_dataset):,}."
    )

# Verify the canonical MNIST test cardinality so accuracy uses the intended benchmark.
if len(test_dataset) != 10_000:
    raise ValueError(
        f"Expected 10,000 test images; found {len(test_dataset):,}."
    )

sample = train_dataset[0]

# A shape mismatch would invalidate the fixed 784-input MLP architecture.
if sample["image"].shape != (INPUT_DIM,):
    raise ValueError(
        f"Expected flattened image shape {(INPUT_DIM,)}; "
        f"found {sample['image'].shape}."
    )

# Labels outside 0–9 cannot be consumed safely by ten-class CrossEntropyLoss.
if not (0 <= sample["label"] < N_CLASSES):
    raise ValueError("MNIST label is outside the expected range 0–9.")

# The MLP comparison assumes identical [0,1] input scaling for every sample.
if (
    sample["image"].min() < 0.0
    or sample["image"].max() > 1.0
):
    raise ValueError("MNIST pixels are not normalized to [0, 1].")

print(f"Training images: {len(train_dataset):,}")
print(f"Testing images: {len(test_dataset):,}")
print(f"Flattened image shape: {sample['image'].shape}")
print(f"Pixel range: [{sample['image'].min():.3f}, {sample['image'].max():.3f}]")


Training images: 60,000
Testing images: 10,000
Flattened image shape: (784,)
Pixel range: [0.000, 1.000]


## 4. Visualize Representative MNIST Samples


In [5]:
# Look at the data before learning from it — a flipped label or malformed image is cheaper to catch now than after training.
sample_indices = np.arange(12)

fig, axes = plt.subplots(
    3,
    4,
    figsize=(8, 6),
)

axes = axes.reshape(-1)

for ax, index in zip(
    axes,
    sample_indices,
):
    item = train_dataset[index]

    ax.imshow(
        item["image"].reshape(
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        cmap="gray",
    )

    ax.set_title(
        f"Label: {item['label']}"
    )
    ax.axis("off")

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR / "mnist_sample_batch.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 5. Define the One-Hidden-Layer MLP Classifier


In [6]:
# Width is the only architecture knob we want to study, so everything else stays fixed.
class MNISTClassifier(nn.Module):
    """Keep the network simple so hidden width is the variable we actually study.

    Every model follows the same path:

        784 inputs -> hidden layer -> BatchNorm -> ReLU -> 10 logits

    Only the hidden width changes between 128, 256, and 512 neurons.

    We deliberately stop at logits. CrossEntropyLoss already applies the
    appropriate log-softmax internally, so adding Softmax here would be both
    redundant and numerically less stable.
    """

    def __init__(
        self,
        hidden_size,
        nclasses=N_CLASSES,
        imsize=IMAGE_SIZE,
        use_batch_norm=True,
    ):
        """Assemble one version of the shared architecture.

        hidden_size is the experimental knob. Batch normalization, activation,
        output size, and loss stay fixed so a later performance difference can
        reasonably be attributed to capacity rather than a different recipe.
        """
        super().__init__()

        layers = [
            nn.Linear(
                imsize * imsize,
                hidden_size,
            ),
        ]

        # BatchNorm keeps hidden activations well-behaved — and stays identical for every width.
        if use_batch_norm:
            layers.append(
                nn.BatchNorm1d(
                    hidden_size
                )
            )

        # ReLU gives the network non-linearity; the last layer returns one raw score per digit.
        layers.extend(
            [
                nn.ReLU(),
                nn.Linear(
                    hidden_size,
                    nclasses,
                ),
            ]
        )

        self.net = nn.Sequential(*layers)

        # Do not add Softmax here: CrossEntropyLoss already handles the numerically stable version internally.
        self.loss_fn = nn.functional.cross_entropy

    def forward(self, data_dict):
        """Use the same network differently depending on whether we train or evaluate.

        During training we need logits plus a differentiable loss so gradients
        can flow backward.

        During evaluation there is no optimizer step. We decode the logits into
        a predicted class and confidence instead.

        One forward method supports both cases without changing the underlying
        architecture.
        """
        logits = self.net(
            data_dict["image"]
        )

        # Training asks, "How wrong am I?" Evaluation asks, "Which class do I choose?"
        if self.training:
            loss = self.loss_fn(
                logits,
                data_dict["label"],
            )

            return {
                "loss": loss,
                "logits": logits,
            }

        predicted_class, confidence = (
            self.decode_prediction(logits)
        )

        return {
            "cls": predicted_class,
            "prob": confidence,
            "logits": logits,
        }

    @staticmethod
    def decode_prediction(logits):
        """Turn raw class scores into something a human can read.

        Softmax converts logits into a probability distribution. We then keep
        the most probable digit and its probability as a simple confidence
        score.

        Softmax appears here — after training loss — because CrossEntropyLoss
        wants the raw logits directly.
        """
        # Softmax is applied only for interpretation/selection, never before training loss.
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        confidence, predicted_class = torch.max(
            probabilities,
            dim=1,
        )

        return predicted_class, confidence

print("MNISTClassifier defined.")

MNISTClassifier defined.


## 6. Verify the Model Architecture and Forward Pass


In [7]:
# Run one mini-batch first; if shapes or loss are wrong, ten epochs will only waste time.
# Use the middle-width network as a quick architecture sanity check before full training.
example_model = MNISTClassifier(
    hidden_size=256,
).to(device)

print(example_model)

# Pull one real training batch so shape/loss validation matches the actual data pipeline.
example_batch = next(
    iter(train_loader)
)

example_batch = {
    "image": torch.from_numpy(
        example_batch["image"]
    ).to(device),
    "label": torch.from_numpy(
        example_batch["label"]
    ).to(device),
}

example_model.train()
# Run one forward pass before optimization to catch architecture or dtype problems early.
example_output = example_model(
    example_batch
)

# A NaN or infinite loss means the numerical pipeline is already broken — stop before optimization starts.
if not torch.isfinite(
    example_output["loss"]
):
    raise ValueError(
        "Example forward pass produced a non-finite loss."
    )

expected_shape = (
    example_batch["image"].shape[0],
    N_CLASSES,
)

# The final layer must emit one score per class for every sample in the batch.
if tuple(
    example_output["logits"].shape
) != expected_shape:
    raise ValueError(
        "Unexpected logit shape: "
        f"{tuple(example_output['logits'].shape)}"
    )

print(
    f"Example training loss: "
    f"{example_output['loss'].item():.4f}"
)
print(
    f"Logit shape: "
    f"{tuple(example_output['logits'].shape)}"
)


MNISTClassifier(
  (net): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)
Example training loss: 2.3383
Logit shape: (512, 10)


## 7. Define Training and Evaluation Utilities


In [8]:
def prepare_batch(
    batch,
    target_device,
):
    """Cross the boundary from NumPy data to PyTorch tensors.

    The Dataset stays simple and device-agnostic. Only when a batch is about to
    enter the model do we convert it and move it to CPU or GPU.

    That keeps data loading independent from the hardware used for training.
    """
    return {
        "image": torch.from_numpy(
            batch["image"]
        ).to(target_device),
        "label": torch.from_numpy(
            batch["label"]
        ).to(target_device),
    }


def train_model(
    model,
    train_loader,
    optimizer,
    epochs,
    target_device,
):
    """Repeat the basic learning loop until the requested epochs are complete.

    Each mini-batch follows the same rhythm:

        clear old gradients -> forward pass -> loss -> backward pass -> update

    We accumulate loss weighted by the actual batch size. That matters because
    the final batch may be smaller than the others; giving every batch equal
    weight would slightly bias the epoch average.
    """
    # Store one mean loss per epoch so convergence can be compared across hidden widths.
epoch_losses = []

    for epoch in range(epochs):
        # Enable BatchNorm training behavior before optimization begins.
        model.train()

        # Accumulate loss weighted by sample count instead of averaging batch means equally.
running_loss = 0.0
        # Track the true number of samples contributing to the epoch loss denominator.
sample_count = 0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
            leave=False,
        )

        for batch in progress:
            batch = prepare_batch(
                batch,
                target_device,
            )

            # PyTorch accumulates gradients by default, so clear the previous batch before learning from this one.
            optimizer.zero_grad()

            output = model(batch)
            loss = output["loss"]

            # Backpropagation turns the current loss into a gradient for every trainable parameter.
            loss.backward()
            # Now that gradients exist, let Adam take one step in the direction that reduces the loss.
            optimizer.step()

            # The final mini-batch may be smaller than 512, so use its actual size for weighting.
current_batch_size = (
                batch["image"].shape[0]
            )

            running_loss += (
                loss.item()
                * current_batch_size
            )

            sample_count += (
                current_batch_size
            )

        # Convert accumulated sample-weighted loss into one comparable epoch statistic.
mean_loss = (
            running_loss
            / sample_count
        )

        epoch_losses.append(
            mean_loss
        )

    return epoch_losses


@torch.no_grad()
def evaluate_model(
    model,
    data_loader,
    target_device,
):
    """Measure what the trained network does when learning is switched off.

    model.eval() freezes training-specific behavior such as BatchNorm updates.
    torch.no_grad() stops PyTorch from building a gradient graph we will never
    use.

    We keep predictions, confidence, labels, and logits because later sections
    need more than one final accuracy number.
    """
    # Evaluation should measure the learned model, not keep changing BatchNorm statistics.
    model.eval()

    all_classes = []
    # Collect confidence values across batches so selective-prediction analysis uses the full test set.
all_probabilities = []
    # Retain ground-truth labels in the same order as predictions for exact accuracy checks.
all_labels = []
    # Preserve raw logits for probability validation and later diagnostics without rerunning inference.
all_logits = []

    for batch in data_loader:
        batch = prepare_batch(
            batch,
            target_device,
        )

        output = model(batch)

        all_classes.append(
            output["cls"].cpu()
        )
        all_probabilities.append(
            output["prob"].cpu()
        )
        all_labels.append(
            batch["label"].cpu()
        )
        all_logits.append(
            output["logits"].cpu()
        )

    classes = torch.cat(
        all_classes
    )
    # Concatenate batch confidences into one test-set vector before computing global diagnostics.
probabilities = torch.cat(
        all_probabilities
    )
    # Concatenate labels in the same batch order so every prediction keeps its correct target.
labels = torch.cat(
        all_labels
    )
    # Preserve the full test-set score matrix for later probability and shape validation.
logits = torch.cat(
        all_logits
    )

    # Compute global test accuracy only after concatenating all batch-level predictions.
accuracy = (
        classes == labels
    ).float().mean().item()

    return {
        "cls": classes,
        "prob": probabilities,
        "label": labels,
        "logits": logits,
        "accuracy": accuracy,
    }

print("Training and evaluation utilities defined.")

Training and evaluation utilities defined.


## 8. Train the 128-, 256-, and 512-Neuron Models


In [9]:
models = {}
training_histories = {}
# Keep one complete evaluation record per hidden width so comparison uses identical metrics.
evaluation_results = {}

for hidden_size in HIDDEN_SIZES:
    print(
        f"\nTraining model: "
        f"{INPUT_DIM} → {hidden_size} → {N_CLASSES}"
    )

    # Reset initialization so model-width comparisons remain controlled.
    torch.manual_seed(SEED)

    model = MNISTClassifier(
        hidden_size=hidden_size,
    ).to(device)

    # Give every width the same optimizer and regularization so capacity remains the only intended difference.
    # the only intended experimental difference.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    losses = train_model(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        epochs=EPOCHS,
        target_device=device,
    )

    evaluation = evaluate_model(
        model=model,
        data_loader=test_loader,
        target_device=device,
    )

    models[hidden_size] = model
    training_histories[hidden_size] = losses
    evaluation_results[hidden_size] = evaluation

    print(
        f"Final training loss: "
        f"{losses[-1]:.4f}"
    )
    print(
        f"Test accuracy: "
        f"{evaluation['accuracy']:.4f}"
    )



Training model: 784 → 128 → 10


Final training loss: 0.0795
Test accuracy: 0.9718

Training model: 784 → 256 → 10


Final training loss: 0.0743
Test accuracy: 0.9718

Training model: 784 → 512 → 10


Final training loss: 0.0717
Test accuracy: 0.9726


## 9. Compare Training Loss and Test Accuracy


In [10]:
# Put all loss curves on the same axes so convergence speed and final fit are directly comparable.
fig, ax = plt.subplots(
    figsize=(9, 5)
)

for hidden_size in HIDDEN_SIZES:
    losses = training_histories[
        hidden_size
    ]

    ax.plot(
        range(1, EPOCHS + 1),
        losses,
        marker="o",
        label=(
            f"{hidden_size} hidden neurons"
        ),
    )

ax.set_title(
    "Training Loss by Hidden-Layer Size"
)
ax.set_xlabel("Epoch")
ax.set_ylabel(
    "Mean Cross-Entropy Loss"
)
ax.set_xticks(
    range(1, EPOCHS + 1)
)
ax.grid(alpha=0.25)
ax.legend()

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "training_loss_comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

# Reference losses are a reality check, not numbers the experiment is supposed to imitate.
REFERENCE_REPORTED_LOSSES = {
    128: 0.0753,
    256: 0.0344,
    512: 0.0394,
}

print(
    f"{'Hidden':>8} "
    f"{'Final loss':>12} "
    f"{'Test accuracy':>15} "
    f"{'Reference loss':>15}"
)
print("-" * 55)

for hidden_size in HIDDEN_SIZES:
    # Use the last recorded training loss only because the lab's comparison rule explicitly asks for it.
final_loss = (
        training_histories[
            hidden_size
        ][-1]
    )

    test_# Compute global test accuracy only after concatenating all batch-level predictions.
accuracy = (
        evaluation_results[
            hidden_size
        ]["accuracy"]
    )

    print(
        f"{hidden_size:>8d} "
        f"{final_loss:>12.4f} "
        f"{test_accuracy:>15.4f} "
        f"{REFERENCE_REPORTED_LOSSES[hidden_size]:>15.4f}"
    )


  Hidden   Final loss   Test accuracy  Reference loss
-------------------------------------------------------
     128       0.0795          0.9718          0.0753
     256       0.0743          0.9718          0.0344
     512       0.0717          0.9726          0.0394


## 10. Visualize Predictions and Confidence Scores


In [11]:
# Pair predictions with full class probabilities to inspect confidence, not only labels.
def plot_prediction_examples(
    model,
    dataset,
    hidden_size,
    output_path,
    n_images=15,
):
    """Look beyond accuracy and inspect how the model arrives at its predictions.

    For the same fixed test examples, we show the digit, the predicted label,
    the true label, and the full probability distribution over all ten classes.

    Using the same examples for every hidden width makes visual comparison fair.
    This is a diagnostic view — not a substitute for the full test-set metric.
    """
    model.eval()

    # Stack individual normalized samples into one contiguous mini-batch array.
images = np.stack(
        [
            dataset[index]["image"]
            for index in range(n_images)
        ]
    ).astype(np.float32)

    # Keep labels as int64 class indices because CrossEntropyLoss consumes integer targets.
labels = np.asarray(
        [
            dataset[index]["label"]
            for index in range(n_images)
        ],
        dtype=np.int64,
    )

    batch = {
        "image": torch.from_numpy(
            images
        ).to(device),
        "label": torch.from_numpy(
            labels
        ).to(device),
    }

    with torch.no_grad():
        output = model(batch)

    probabilities = torch.softmax(
        output["logits"],
        dim=1,
    ).cpu().numpy()

    fig = plt.figure(
        figsize=(12, 10)
    )

    for index in range(n_images):
        image_axis = plt.subplot(
            5,
            6,
            2 * index + 1,
        )

        image_axis.imshow(
            images[index].reshape(
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            cmap="gray",
        )

        predicted_label = int(
            np.argmax(
                probabilities[index]
            )
        )

        confidence = float(
            np.max(
                probabilities[index]
            )
        )

        true_label = int(
            labels[index]
        )

        image_axis.set_xlabel(
            f"Pred {predicted_label} "
            f"({confidence:.0%})\n"
            f"True {true_label}"
        )

        image_axis.set_xticks([])
        image_axis.set_yticks([])

        value_axis = plt.subplot(
            5,
            6,
            2 * index + 2,
        )

        value_axis.bar(
            range(N_CLASSES),
            probabilities[index],
        )

        value_axis.set_ylim(
            0,
            1,
        )
        value_axis.set_xticks(
            range(N_CLASSES)
        )
        value_axis.set_yticks([])

    fig.suptitle(
        f"MNIST Predictions — "
        f"{hidden_size} Hidden Neurons",
        y=1.01,
    )

    plt.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


for hidden_size in HIDDEN_SIZES:
    plot_prediction_examples(
        model=models[hidden_size],
        dataset=test_dataset,
        hidden_size=hidden_size,
        output_path=(
            OUTPUT_DIR
            / f"mnist_predictions_{hidden_size}.png"
        ),
    )

## 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy


In [12]:
def compute_confidence_curves(
    evaluation,
    # One hundred thresholds provide a smooth confidence curve without excessive redundant samples.
n_thresholds=100,
):
    """Ask what happens if the model is allowed to say, "I'm not confident enough."

    For each confidence threshold, we keep only predictions above that level.
    Raising the threshold usually improves reliability among accepted cases but
    reduces how many useful predictions survive.

    The resulting precision/recall-like curves describe selective prediction,
    not the usual per-class precision-recall problem.
    """
    predicted_classes = (
        evaluation["cls"].numpy()
    )

    confidence = (
        evaluation["prob"].numpy()
    )

    labels = (
        evaluation["label"].numpy()
    )

    correct = (
        predicted_classes
        == labels
    )

    # Sweep confidence from permissive to strict acceptance to expose the coverage/reliability trade-off.
thresholds = np.linspace(
        0.0,
        0.99,
        n_thresholds,
    )

    precision_values = []
    recall_values = []
    # Record correctness only among accepted predictions at each confidence threshold.
accepted_accuracy_values = []

    total_correct = np.sum(
        correct
    )

    for threshold in thresholds:
        # Confidence does not mean "correct"; it only decides whether this prediction is accepted.
        accepted = (
            confidence
            > threshold
        )

        true_positive = np.sum(
            accepted & correct
        )

        false_positive = np.sum(
            accepted & ~correct
        )

        false_negative = np.sum(
            ~accepted & correct
        )

        precision_denominator = (
            true_positive
            + false_positive
        )

        recall_denominator = (
            true_positive
            + false_negative
        )

        # No accepted predictions means accepted precision is undefined, not zero.
        if precision_denominator > 0:
            precision = true_positive / precision_denominator
        else:
            precision = np.nan

        # If no correct predictions exist to retain, selective recall is defined as zero.
        if recall_denominator > 0:
            recall = true_positive / recall_denominator
        else:
            recall = 0.0

        accepted_# Compute global test accuracy only after concatenating all batch-level predictions.
accuracy = (
            true_positive
            / len(labels)
        )

        precision_values.append(
            precision
        )
        recall_values.append(
            recall
        )
        accepted_accuracy_values.append(
            accepted_accuracy
        )

    precision_values = np.asarray(
        precision_values
    )
    recall_values = np.asarray(
        recall_values
    )
    accepted_accuracy_values = np.asarray(
        accepted_accuracy_values
    )

    # Sort thresholds before plotting so all selective-classification curves progress monotonically in x.
order = np.argsort(
        recall_values
    )

    return {
        "thresholds": thresholds[order],
        "precision": precision_values[order],
        "recall": recall_values[order],
        "accepted_accuracy": accepted_accuracy_values[order],
        "total_correct": int(total_correct),
    }

print("Confidence-threshold evaluation utility defined.")


Confidence-threshold evaluation utility defined.


## 12. Analyze the Best Model and Save Evaluation Curves


In [13]:
# The assignment asks us to choose by final training loss, so apply that rule explicitly — even though validation loss would be better practice.
# This is kept for assignment alignment; in general, model selection should rely
# on validation performance rather than training loss alone.
best_hidden_size = min(
    HIDDEN_SIZES,
    key=lambda size: (
        training_histories[
            size
        ][-1]
    ),
)

best_evaluation = (
    evaluation_results[
        best_hidden_size
    ]
)

# Compute the confidence trade-off once for the selected model and reuse it for both plots.
confidence_curves = (
    compute_confidence_curves(
        best_evaluation,
        # One hundred thresholds provide a smooth confidence curve without excessive redundant samples.
n_thresholds=100,
    )
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
)

axes[0].plot(
    confidence_curves["recall"],
    confidence_curves["precision"],
)
axes[0].set_title(
    "Precision–Recall Curve"
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].grid(alpha=0.25)

axes[1].plot(
    confidence_curves["recall"],
    confidence_curves[
        "accepted_accuracy"
    ],
)
axes[1].set_title(
    "Accepted Accuracy–Recall Curve"
)
axes[1].set_xlabel("Recall")
axes[1].set_ylabel(
    "Accepted correct / all test samples"
)
axes[1].grid(alpha=0.25)

fig.suptitle(
    f"Confidence-Threshold Evaluation — "
    f"{best_hidden_size} Hidden Neurons"
)

plt.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "pr_accuracy_curve.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(
    f"Selected hidden size: "
    f"{best_hidden_size}"
)
print(
    f"Final training loss: "
    f"{training_histories[best_hidden_size][-1]:.4f}"
)
print(
    f"Test accuracy: "
    f"{best_evaluation['accuracy']:.4f}"
)

Selected hidden size: 512
Final training loss: 0.0717
Test accuracy: 0.9726


## 13. Run Numerical and Output-file Validation Checks


In [14]:
# A good accuracy number is not enough; dataset size, histories, probabilities, and outputs must all still be coherent.
# Verify the canonical MNIST training cardinality before comparing models.
if len(train_dataset) != 60_000:
    raise ValueError(
        "Training dataset size changed unexpectedly."
    )

# Verify the canonical MNIST test cardinality so accuracy uses the intended benchmark.
if len(test_dataset) != 10_000:
    raise ValueError(
        "Test dataset size changed unexpectedly."
    )

# Require every planned width so the capacity comparison is complete and fair.
if set(models) != set(HIDDEN_SIZES):
    raise ValueError(
        "Not all required MLP configurations were trained."
    )

for hidden_size in HIDDEN_SIZES:
    # Convert stored Python loss history to a numeric array before finiteness/shape validation.
losses = np.asarray(
        training_histories[
            hidden_size
        ],
        dtype=float,
    )

    # Each width must report exactly one mean loss per requested epoch.
    if losses.shape != (EPOCHS,):
        raise ValueError(
            f"Unexpected loss-history length for hidden size {hidden_size}."
        )

    # Non-finite loss history indicates numerical failure even if training completed.
    if not np.all(
        np.isfinite(losses)
    ):
        raise ValueError(
            f"Non-finite training loss for hidden size {hidden_size}."
        )

    evaluation = (
        evaluation_results[
            hidden_size
        ]
    )

    # Accuracy is a probability-like proportion and must stay inside [0,1].
    if not (
        0.0
        <= evaluation["accuracy"]
        <= 1.0
    ):
        raise ValueError(
            f"Invalid test accuracy for hidden size {hidden_size}."
        )

    # Require one prediction per test sample before trusting reported accuracy.
    if (
        evaluation["cls"].shape[0]
        != len(test_dataset)
    ):
        raise ValueError(
            f"Incomplete test predictions for hidden size {hidden_size}."
        )

    probabilities = (
        evaluation["prob"].numpy()
    )

    # Confidence analysis is invalid if any probability is NaN or infinite.
    if not np.all(
        np.isfinite(probabilities)
    ):
        raise ValueError(
            f"Non-finite confidence values for hidden size {hidden_size}."
        )

    # Softmax-derived confidence must respect the probability interval [0,1].
    if (
        np.any(probabilities < 0.0)
        or np.any(probabilities > 1.0)
    ):
        raise ValueError(
            f"Confidence values outside [0, 1] for hidden size {hidden_size}."
        )

# Declare the full diagnostic contract so a run is not considered complete with missing figures.
REQUIRED_OUTPUTS = [
    "mnist_sample_batch.png",
    "training_loss_comparison.png",
    "mnist_predictions_128.png",
    "mnist_predictions_256.png",
    "mnist_predictions_512.png",
    "pr_accuracy_curve.png",
]

# If one expected figure is missing, the experiment is not fully reproducible from the notebook.
missing_outputs = [
    name
    for name in REQUIRED_OUTPUTS
    if not (
        OUTPUT_DIR / name
    ).exists()
]

# Diagnostics are part of the evidence, so a missing file counts as an incomplete run.
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: "
        + ", ".join(
            missing_outputs
        )
    )

print(
    "All Deep Learning validation checks passed."
)


All Deep Learning validation checks passed.
